In [2]:
from azure.cognitiveservices.vision.computervision import ComputerVisionClient
from azure.cognitiveservices.vision.computervision.models import OperationStatusCodes
from azure.cognitiveservices.vision.computervision.models import VisualFeatureTypes
from msrest.authentication import CognitiveServicesCredentials
from array import array
import os
from PIL import Image
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

'''
Authenticate
Authenticates your credentials and creates a client.
'''
subscription_key = os.environ["VISION_KEY"]
endpoint = os.environ["VISION_ENDPOINT"]
computervision_client = ComputerVisionClient(endpoint, CognitiveServicesCredentials(subscription_key))
'''
END - Authenticate
'''

img = open("test2.jpeg", "rb")
read_response = computervision_client.read_in_stream(
    image=img,
    mode="Printed",
    raw=True
)
# print(read_response.as_dict())

operation_id = read_response.headers['Operation-Location'].split('/')[-1]
while True:
    read_result = computervision_client.get_read_result(operation_id)
    if read_result.status not in ['notStarted', 'running']:
        break
    time.sleep(1)

# Print the detected text, line by line
result = []
if read_result.status == OperationStatusCodes.succeeded:
    for text_result in read_result.analyze_result.read_results:
        for line in text_result.lines:
            print(line.text)
            result.append(line.text)

print()

Lucces in resolvarea
TEMELOR la
LABORA toarele de
Inteligenta Artificialà!


In [3]:
# get/define the ground truth
# groundTruth = ["Google Cloud", "Platform"]
groundTruth = ["Succes in rezolvarea", "tEMELOR la", "LABORAtoaree de", "Inteligenta Artificiala!"]

# compute the performance
noOfCorrectLines = sum(i == j for i, j in zip(result, groundTruth))
print(noOfCorrectLines)

0


In [4]:
img_path = "ocr.png"
img = open(img_path, "rb")
image = Image.open(img_path)
img_width, img_height = image.width, image.height

# Recunoașterea textului scris de mână
read_response = computervision_client.read_in_stream(
    image=img,
    mode="Handwritten",  # Modificat pentru a recunoaște textul scris de mână
    raw=True
)

operation_id = read_response.headers['Operation-Location'].split('/')[-1]

# Așteptare până când procesul de recunoaștere este complet
while True:
    read_result = computervision_client.get_read_result(operation_id)
    if read_result.status not in ['notStarted', 'running']:
        break
    time.sleep(1)


def get_centroid(bounding_box):
    x_coords = bounding_box[0::2]
    y_coords = bounding_box[1::2]
    centroid_x = sum(x_coords) / len(x_coords)
    centroid_y = sum(y_coords) / len(y_coords)
    return centroid_x, centroid_y


def get_location(centroid_x, centroid_y, img_width, img_height):
    vertical = "sus" if centroid_y < img_height / 3 else "jos" if centroid_y > 2 * img_height / 3 else "mijloc"
    horizontal = "stânga" if centroid_x < img_width / 3 else "dreapta" if centroid_x > 2 * img_width / 3 else "centru"
    
    if vertical == "mijloc" and horizontal == "centru":
        return "centru"
    return f"{horizontal}-{vertical}"


if read_result.status == OperationStatusCodes.succeeded:
    for text_result in read_result.analyze_result.read_results:
        for line in text_result.lines:
            centroid_x, centroid_y = get_centroid(line.bounding_box)
            location = get_location(centroid_x, centroid_y, img_width, img_height)
            print(f"Text: {line.text} | Poziție: {location}")

Text: Optical | Poziție: stânga-sus
Text: Character | Poziție: centru-sus
Text: Optical | Poziție: stânga-mijloc
Text: Character | Poziție: centru
Text: Recognition | Poziție: centru
Text: Recognition | Poziție: centru


In [5]:
from rapidfuzz import fuzz, distance

ground_truth = ["Succes in rezolvarea", "tEMELOR la", "LABORAtoaree de", "Inteligenta Artificiala!"]

ocr_results = ["Lucces in rezolvarea", "tEMELOR la", "LABORAtoaree de", "Inteligenta Artificiala!"]

# Calculăm scorurile pentru fiecare text detectat
for gt, ocr in zip(ground_truth, ocr_results):
    cer = distance.Levenshtein.normalized_distance(gt, ocr)
    wer = distance.DamerauLevenshtein.normalized_distance(gt.split(), ocr.split())
    similarity = fuzz.ratio(gt, ocr)

    print(f"Text corect: {gt} | OCR: {ocr}")
    print(f"  CER (Character Error Rate): {cer:.2%}")
    print(f"  WER (Word Error Rate): {wer:.2%}")
    print(f"  Similaritate (fuzz ratio): {similarity:.2f}%\n")

Text corect: Succes in rezolvarea | OCR: Lucces in rezolvarea
  CER (Character Error Rate): 5.00%
  WER (Word Error Rate): 33.33%
  Similaritate (fuzz ratio): 95.00%

Text corect: tEMELOR la | OCR: tEMELOR la
  CER (Character Error Rate): 0.00%
  WER (Word Error Rate): 0.00%
  Similaritate (fuzz ratio): 100.00%

Text corect: LABORAtoaree de | OCR: LABORAtoaree de
  CER (Character Error Rate): 0.00%
  WER (Word Error Rate): 0.00%
  Similaritate (fuzz ratio): 100.00%

Text corect: Inteligenta Artificiala! | OCR: Inteligenta Artificiala!
  CER (Character Error Rate): 0.00%
  WER (Word Error Rate): 0.00%
  Similaritate (fuzz ratio): 100.00%
